In [ ]:
import os
from langchain_community.document_loaders import PyMuPDFLoader

def load_documents():
    folder_path = "data"

    documents = []

    for file in os.listdir(folder_path):

        if file.endswith(".pdf"):

            print(f"Loading: {file}")

            loader = PyMuPDFLoader(os.path.join(folder_path, file))

            docs = loader.load()

            documents.extend(docs)

    print(f"\n✅ Total Pages Loaded: {len(documents)}")

    return documents

In [18]:
# Required Libraries
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

In [19]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print(f"Total Chunks: {len(chunks)}")

Total Chunks: 831


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from src.loader import load_documents
documents = load_documents()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)
print(len(chunks))

Loading: jesc101.pdf
Loading: jesc102.pdf
Loading: jesc103.pdf
Loading: jesc104.pdf
Loading: jesc105.pdf
Loading: jesc106.pdf
Loading: jesc107.pdf
Loading: jesc108.pdf
Loading: jesc109.pdf
Loading: jesc110.pdf
Loading: jesc111.pdf
Loading: jesc112.pdf
Loading: jesc113.pdf
Loading: jesc1an.pdf
Loading: jesc1ps.pdf

✅ Total Pages Loaded: 232
831


In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6325.23it/s]


In [3]:
from langchain_community.vectorstores import Chroma

from src.splitter import split_documents
from src.embeddings import load_embeddings


def create_vector_db():

    chunks = split_documents()

    embeddings = load_embeddings()

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    return vector_store

In [3]:
from src.vectordb import create_vector_db
vector_store=create_vector_db()
def retriever():
    retriever=vector_store.as_retriever(
    search_kwargs={"k":5}
)
    return retriever

Loading: jesc101.pdf
Loading: jesc102.pdf
Loading: jesc103.pdf
Loading: jesc104.pdf
Loading: jesc105.pdf
Loading: jesc106.pdf
Loading: jesc107.pdf
Loading: jesc108.pdf
Loading: jesc109.pdf
Loading: jesc110.pdf
Loading: jesc111.pdf
Loading: jesc112.pdf
Loading: jesc113.pdf
Loading: jesc1an.pdf
Loading: jesc1ps.pdf

✅ Total Pages Loaded: 232


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3283.45it/s]


In [4]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

def load_llm():

    load_dotenv()

    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=GOOGLE_API_KEY
    )

    return llm

In [5]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template("""
You are an experienced CBSE Class 10 Science Teacher.

You MUST answer ONLY from the provided NCERT context.

Follow these rules carefully:

1. If the question asks for a definition:
   • Give the NCERT definition.

2. If the question asks "Explain":
   • Explain step-by-step.
   • Use simple English suitable for Class 10.
   • If possible, give one real-life example.

3. If the question asks "Differentiate":
   • Present the answer in a comparison table.

4. If the question asks "Why":
   • Give the reason first.
   • Then explain.

5. If the question contains MCQ options:
   • First write the correct option.
   • Then explain why it is correct.
   • If a chemical equation exists in the context,
     include it.

6. If the question asks for a chemical reaction:
   • Write the balanced chemical equation.
   • Explain it in simple words.

7. Keep answers concise but complete.

8. Never invent information.

9. If the answer is not present in the NCERT context, reply:

"I couldn't find this information in the provided NCERT documents."

Context:
{context}

Question:
{question}

Answer:
""")

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from src.retriever import retriever
from src.prompt import load_prompt
from src.llm import load_llm
def create_rag_chain():

    retriever_obj = retriever()

    prompt = load_prompt()

    llm = load_llm()

    ragchain = (
        {
            "context": retriever_obj,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return ragchain

In [2]:
from src.vectordb import create_vector_db

def retriever():
    vector_store=create_vector_db()
    retriever=vector_store.as_retriever(
    search_kwargs={"k":5}
)
    return retriever

In [ ]:
print('x')

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from src.retriever import retriever
from src.prompt import load_prompt
from src.llm import load_llm
def create_rag_chain():

    retriever_obj = retriever()

    prompt = load_prompt()

    llm = load_llm()

    ragchain = (
        {
            "context": retriever_obj,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return ragchain

Loading: jesc101.pdf
Loading: jesc102.pdf
Loading: jesc103.pdf
Loading: jesc104.pdf
Loading: jesc105.pdf
Loading: jesc106.pdf
Loading: jesc107.pdf
Loading: jesc108.pdf
Loading: jesc109.pdf
Loading: jesc110.pdf
Loading: jesc111.pdf
Loading: jesc112.pdf
Loading: jesc113.pdf
Loading: jesc1an.pdf
Loading: jesc1ps.pdf

✅ Total Pages Loaded: 232


KeyboardInterrupt: 